### Imports

In [3]:
import torchxrayvision as xrv
import torch
from torchxrayvision.datasets import NIH_Dataset
import torchvision.transforms as T

c:\Users\barte\OneDrive\Pulpit\dnn-cxr-diagnostics\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


### Testing library

#### Datasets

In [12]:
dataset = NIH_Dataset(
    imgpath="C:\\Users\\barte\\OneDrive\\Pulpit\\dnn-cxr-diagnostics\\cxr-training-pipeline\\data\\01_raw\\images",
    csvpath="USE_INCLUDED_FILE",
    bbox_list_path="USE_INCLUDED_FILE",
)

In [28]:
sample = dataset[0]
print(sample.keys())
print(sample["lab"])
print(sample["lab"].shape)
print(sample['img'])

dict_keys(['idx', 'lab', 'img'])
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(14,)
[[[  598.3373    574.24316   542.1177  ...  -983.84314 -1007.93726
   -1024.     ]
  [  574.24316   550.14905   542.1177  ...  -983.84314 -1007.93726
   -1024.     ]
  [  550.14905   534.0863    526.05493 ...  -983.84314 -1007.93726
   -1024.     ]
  ...
  [ 1024.       1024.       1024.      ... -1024.      -1024.
   -1024.     ]
  [ 1024.       1024.       1015.9686  ... -1024.      -1024.
   -1024.     ]
  [ 1024.       1024.       1024.      ... -1024.      -1024.
   -1024.     ]]]


#### Models

In [4]:
# model trained on nih-cxr dataset
model = xrv.models.DenseNet(weights="densenet121-res224-nih")

In [ ]:
# my thresholds
labels_order = ['Infiltration', 'Effusion', 'Atelectasis', 'Nodule', 'Mass', 'Pneumothorax', 'Consolidation', 'Pleural_Thickening', 'Cardiomegaly', 'Emphysema', 'Edema', 'Fibrosis', 'Pneumonia', 'Hernia']

# as of 09.03
my_thresholds = [0.17000000178813934, 0.10999999940395355, 0.10000000149011612, 0.03999999910593033, 0.03999999910593033, 0.029999999329447746, 0.029999999329447746, 0.019999999552965164, 0.019999999552965164, 0.019999999552965164, 0.019999999552965164, 0.009999999776482582, 0.009999999776482582, 0.0]

xrv_thresholds = {}
my_best_thresholds = {}

# xrv thresholds
for pathology, op_thresh in zip(model.pathologies, model.op_threshs):
    xrv_thresholds[pathology] = op_thresh

for pathology, my_thresh in zip(labels_order, my_thresholds):
    my_best_thresholds[pathology] = my_thresh

for item in xrv_thresholds.keys():
    if item != "":
        print(f"{item}: xrv = {xrv_thresholds[item]:.6f} | my_threshold = {my_best_thresholds[item]:.6f}" )


Atelectasis: xrv = 0.039118 | my_threshold = 0.100000
Consolidation: xrv = 0.003453 | my_threshold = 0.030000
Infiltration: xrv = 0.113963 | my_threshold = 0.170000
Pneumothorax: xrv = 0.005730 | my_threshold = 0.030000
Edema: xrv = 0.000457 | my_threshold = 0.020000
Emphysema: xrv = 0.001888 | my_threshold = 0.020000
Fibrosis: xrv = 0.012038 | my_threshold = 0.010000
Effusion: xrv = 0.038744 | my_threshold = 0.110000
Pneumonia: xrv = 0.003721 | my_threshold = 0.010000
Pleural_Thickening: xrv = 0.014731 | my_threshold = 0.020000
Cardiomegaly: xrv = 0.016150 | my_threshold = 0.020000
Nodule: xrv = 0.054241 | my_threshold = 0.040000
Mass: xrv = 0.037199 | my_threshold = 0.040000
Hernia: xrv = 0.000440 | my_threshold = 0.000000


### Predictions

In [13]:
# resize to 224
transform = T.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])

dataset = NIH_Dataset(
    imgpath="C:\\Users\\barte\\OneDrive\\Pulpit\\dnn-cxr-diagnostics\\cxr-training-pipeline\\data\\01_raw\\images",
    transform=transform
)

In [17]:
def predict(model, sample):
    img = torch.from_numpy(sample["img"]).unsqueeze(0)
    with torch.no_grad():
        preds = model(img)

    probs = torch.sigmoid(preds)[0]

    for pathology, prob in zip(model.pathologies, probs):
        print(f"{pathology}: {prob:.3f}")

    labels = sample["lab"]

    for pathology, label in zip(dataset.pathologies, labels):
        if label == 1:
            print("True pathology:", pathology)

#### 1. Model trained only on NIH dataset

In [18]:
model = xrv.models.DenseNet(weights="densenet121-res224-nih")
model.eval()

predict(model, dataset[0])

Atelectasis: 0.628
Consolidation: 0.623
Infiltration: 0.624
Pneumothorax: 0.542
Edema: 0.623
Emphysema: 0.625
Fibrosis: 0.628
Effusion: 0.638
Pneumonia: 0.623
Pleural_Thickening: 0.626
Cardiomegaly: 0.661
Nodule: 0.626
Mass: 0.630
Hernia: 0.623
: 0.622
: 0.622
: 0.622
: 0.622
True pathology: Cardiomegaly


#### 2. CheXpert model

In [19]:
model = xrv.models.DenseNet(weights="densenet121-res224-chex")
model.eval()

predict(model, dataset[0])

Atelectasis: 0.638
Consolidation: 0.565
: 0.622
Pneumothorax: 0.520
Edema: 0.637
: 0.622
: 0.622
Effusion: 0.558
Pneumonia: 0.576
: 0.622
Cardiomegaly: 0.688
: 0.622
: 0.622
: 0.622
Lung Lesion: 0.575
Fracture: 0.631
Lung Opacity: 0.631
Enlarged Cardiomediastinum: 0.635
True pathology: Cardiomegaly


#### 3. Model trained on all datasets

In [20]:
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model.eval()

predict(model, dataset[0])

Atelectasis: 0.624
Consolidation: 0.576
Infiltration: 0.628
Pneumothorax: 0.578
Edema: 0.548
Emphysema: 0.623
Fibrosis: 0.632
Effusion: 0.597
Pneumonia: 0.537
Pleural_Thickening: 0.625
Cardiomegaly: 0.648
Nodule: 0.625
Mass: 0.605
Hernia: 0.503
Lung Lesion: 0.552
Fracture: 0.572
Lung Opacity: 0.597
Enlarged Cardiomediastinum: 0.607
True pathology: Cardiomegaly


#### Conclusions
- In both cases, the model assigned the highest probability to Cardiomegaly, which aligns with the actual diagnosis for this image. 
- The variance in the model's confidence is minimal, with a maximum difference of only 0.040.